In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation

from utils import *

In [2]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

[]

In [3]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file, fringes_file, ghost_file = files[::2]
fringes_file_ = files[3]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(fringes_file) as hdul:
    fringes = hdul[0].data
    header = hdul[0].header
    wv0 = read_wavelengths(header)[header['CONTPOS'] - 1]

with fits.open(fringes_file_) as hdul:
    fringes_ = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

print(wv0)

ghost = demodulate(ghost)
fringes = demodulate(fringes)
fringes_ = demodulate(fringes_)

ValueError: not enough values to unpack (expected 3, got 0)

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(fringes_[3], vmin=-1e-3, vmax=1e-3)
plt.tight_layout()

In [11]:
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T233009_V202405152329C_0443300501.fits.gz',
 '/home/ulyanov/data/solo/ph

In [149]:
with fits.open(files[-5]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

nx, ny = data.shape[-2:]

cpos = header['CONTPOS'] - 1
wv = read_wavelengths(header)
data = data.reshape(6,4,nx,ny)
#data = calc_continuum(data, wv, continuum=cpos)

data -= crop(dark, header) * 0.4
data /= crop(flat, header)
data = realign(data)
data = demodulate(data)

xr, yr = reflection_point_predict(header)
reflection = reflect(gaussian_filter(data[:,0], 8, axes=(-2,-1)), xr, yr)

for i in range(4):
    data[:,i] -= crop(ghost[i], header) * reflection

In [150]:
pol = 2

i = -1
dwv = wv[i] - wv0
dphi = dwv * 25
print(dphi / np.pi)

temp = fringes[pol] * np.cos(dphi) - fringes_[pol] * np.sin(dphi)
temp = crop(temp, header)

plt.figure(figsize=(10,10))
plt.imshow(data[i,pol] - temp * data[i,0] * 0.8, vmin=-30, vmax=30)
plt.tight_layout()

2.0212677772663183
